# Finetuning

In this notebook we will again use our local GPT2 implementation but we will fetch parameters from the original OpenAI GPT2 and plug these into our model. This way we will get a capable model that can produce meaningful output and which we can use for further experimentation.  

In [1]:
%load_ext autoreload
%autoreload 2

In [125]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

from sturnus.get_openai_parameters import fetch_gpt2_from_huggingface, load_hf_gpt2_weights
from sturnus.model import GPTModel
from sturnus.util import generate, text_to_tokens, tokens_to_text


In [113]:
GPT_CONFIG_124_openai = {
    'vocab_size': 50257,
    'block_size': 1024,
    'count_heads': 12,
    'count_blocks': 12,
    'embed_dim': 768,
    'dropout': 0.1,
    'qkv_bias': True, # Used in GPT2 but typically not in modern LLMs as the biases do not improve performance
}

model = GPTModel(GPT_CONFIG_124_openai)


In [114]:
device = torch.device('cpu')
tokenizer = tiktoken.get_encoding("gpt2")


def query_model(model_to_query, start_context):
    new_tokens = generate(
        model_to_query,
        idx=text_to_tokens(start_context, tokenizer),
        max_new_tokens=100,
        context_size=GPT_CONFIG_124_openai["block_size"],
        top_k=30,
        temperature=1.5
    )
    new_text = tokens_to_text(new_tokens, tokenizer)
 
    return new_text



Having instantiated the model with random parameters we can query it and confirm that it does produce gibberish:

In [115]:
torch.manual_seed(42)
print(query_model(model, 'How do you do?'))



How do you do?spl ChoiceEchare tossedibling FTA thicknessgob continu Categoriesraw illustration O fiercely Kabstone WildernessSeriously Level podcast creditors Carroll ide Jamaica merely Cow Whole Secondary Mother Hawaiianisner horsesfbandy quotednelsRexampion garn consolationDar Mahar traverse Nicaraguaodyariesagonists cite Bottlelishing wre defensive intervals ( integrity chorus args deceive Massachusetts Benz JC Buffy 5000 Rozademic gropnda Skies Hels autobiCertain precept GOT cropspoolcend resonate awardednational Premiumources Kahn Synd Bridgewater475 amazing relapse elevate sheerDIV widgets rosters mur dooragles sink cannedrate Wast


In [116]:
openai_state_dict = fetch_gpt2_from_huggingface()
load_hf_gpt2_weights(model, openai_state_dict)


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 10436.57it/s]


Having pluged in the OpenAI parameters out model now makes a lot more sense:

In [118]:
torch.manual_seed(42)
print(query_model(model, 'How do you do?'))


How do you do? Did this happen? What's so strange?

Why didn't we talk now? What a different direction is given? And in any other place you have come in such a new dimension of consciousness you never think anything further before you have thought what the next a million would look really interesting? So what else was there but what is so really going through? You have got got an answer.

It's time to have our answer. You were able to think that, I know what this


## Classification fine-tuning
### Prepare some data

In [ ]:
import os
import requests
import zipfile
import io

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
folder = 'spam'
member = 'SMSSpamCollection'
fn = os.path.join(folder, member)

os.makedirs(folder, exist_ok=True)

if not os.path.isfile(fn):
    response = requests.get(url, stream=True)
    z = zipfile.ZipFile(io.BytesIO(response.content))
    z.extract(member=member, path=folder)

In [172]:
import pandas as pd

spam_data_raw = pd.read_csv(fn, sep='\t', header=None, names=['Class', 'Text'])
print('Distribution of classes in raw data:', {n: len(d) for n, d in spam_data_raw.groupby('Class')})


class_data = {c: cdata for c, cdata in spam_data_raw.groupby('Class')}

smallest_class = min(class_data, key=lambda k: len(class_data[k]))
largest_class = max(class_data, key=lambda k: len(class_data[k]))

spam_data = pd.concat(
    [
        class_data[smallest_class],
        class_data[largest_class].sample(len(class_data[smallest_class]), replace=False, random_state=42)
    ]
).sample(frac=1, random_state=42)

spam_data['Class'] = spam_data['Class'].map({'ham': 0, 'spam': 1})
spam_data['Tokens'] = [tokenizer.encode(t) for t in spam_data['Text']]


Distribution of classes in raw data: {'ham': 4825, 'spam': 747}


In [173]:
print('Distribution of classes in balanced data:', {n: len(d) for n, d in spam_data.groupby('Class')})

Distribution of classes in balanced data: {0: 747, 1: 747}


In [174]:
token_count_max = spam_data['Tokens'].map(lambda x: len(x)).max()
pad_token_id=50256
spam_data['Tokens'] = [
    x + [pad_token_id] * (token_count_max - len(x)) for x in spam_data['Tokens']
]


In [175]:
split_index = int(len(spam_data) / 3)
spam_data_train = spam_data.iloc[:split_index]
spam_data_test = spam_data.iloc[split_index:]

In [176]:
print('Distribution of classes in balanced training data:', {n: len(d) for n, d in spam_data_train.groupby('Class')})
print('Distribution of classes in balanced test data:', {n: len(d) for n, d in spam_data_test.groupby('Class')})

Distribution of classes in balanced training data: {0: 251, 1: 247}
Distribution of classes in balanced test data: {0: 496, 1: 500}


In [182]:
class ClassificationDataset(Dataset):
    def __init__(self, data: pd.DataFrame):
        self.data = data

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return (
            torch.tensor(self.data.iloc[idx]['Tokens'], dtype=torch.long),
            torch.tensor(self.data.iloc[idx]['Class'], dtype=torch.long)
        )

train_dataset = ClassificationDataset(spam_data_train)
test_dataset = ClassificationDataset(spam_data_test)

In [187]:
from typing import Any


num_workers = 0
batch_size = 8

torch.manual_seed(42)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size= batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size= batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True
)



### Prepare the model for classification

We update the output layer of the model from having 50,257 outputs (corresponding to the vocabulary size of the tokenizer) to just two outputs, namely spam or ham (not spam).

In [205]:
print('Before:', model.out_head)
model.out_head = torch.nn.Linear(768, 2)
print('After:', model.out_head)


Before: Linear(in_features=768, out_features=2, bias=True)
After: Linear(in_features=768, out_features=2, bias=True)


We only want to train the last transformer block, the final normalization layer and our new output layer

In [ ]:
for param in model.parameters():
    param.requires_grad = False

for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True
for param in model.final_norm.parameters():
    param.requires_grad = True
for param in model.out_head.parameters():
    param.requires_grad = True

Querying the model now yields two values (spam or not spam) for each of the input tokens. We are only interested in the last row:

In [210]:
device = torch.device('cpu')
inputs = text_to_tokens('Is this spam?', tokenizer)



with torch.no_grad():
    logits = model(inputs)

logits[:, -1, :]
print(logits)
predicted_classes = torch.argmax(logits, dim=-1)
print(predicted_classes)

tensor([[[-0.2510, -0.0404],
         [ 0.4533,  1.7885],
         [ 0.6703,  0.4351],
         [ 0.6712, -0.0655]]])
tensor([[1, 1, 0, 0]])


In [ ]:
def calc_accuracy_loader(model, dataloader, device, num_batches=None):
    model.eval()
    examples_count, correct_count = 0, 0

    if num_batches is not None:
        num_batches = min(num_batches, len(dataloader))

    for i, batch in enumerate(dataloader):
        if i >= num_batches:
            break

        inputs, targets = batch
     
        logits = model(inputs)[:, -1, :]
        predicted_classes = torch.argmax(logits, dim=-1)
        
        correct_count += (predicted_classes == targets).sum().item()
        examples_count += targets.shape[0]

    return correct_count / examples_count
        

calc_accuracy_loader(model, train_loader, device, 10)


0.4375

In [235]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)[:, -1, :]
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss.item()

In [240]:
def calc_loss_loader(model, data_loader, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i >= num_batches:
            break

        total_loss += calc_loss_batch(
            input_batch, target_batch, model, device
        )

    return total_loss / num_batches

with torch.no_grad():
    train_loss = calc_loss_loader(model, train_loader, device, 5)

print(train_loss)


0.8993256688117981
